In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Carga del dataset
df = pd.read_csv("/workspace/Housing.csv")

print("Primeras filas:")
display(df.head())

print("Información del DataFrame:")
df.info()

print("Valores nulos:")
print(df.isnull().sum())

print("Descripción estadística:")
display(df.describe())

# 2. Exploración y visualización
plt.figure(figsize=(8,4))
sns.histplot(df["price"], kde=True)
plt.title("Histograma de price")
plt.xlabel("price")
plt.ylabel("Frecuencia")
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(x=df["price"])
plt.title("Boxplot de price")
plt.xlabel("price")
plt.show()

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
sns.pairplot(df[numeric_cols])
plt.show()

plt.figure(figsize=(8,6))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Mapa de calor de correlación")
plt.show()

# 3. Preprocesamiento
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

print("Valores nulos después del tratamiento:")
print(df.isnull().sum())

categorical_cols = df.select_dtypes(include=["object"]).columns
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Primeras filas tras get_dummies:")
display(df_encoded.head())

X = df_encoded.drop("price", axis=1)
y = df_encoded["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Modelo de regresión
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print("Intercepto del modelo:")
print(model.intercept_)

coeficientes = pd.DataFrame({
    "variable": X.columns,
    "coeficiente": model.coef_
}).sort_values(by="coeficiente", ascending=False)

print("Coeficientes del modelo:")
display(coeficientes)

# 5. Predicciones y evaluación
y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("MAE:", mae)
print("R2:", r2)

# 6. Visualización de resultados
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred)
plt.xlabel("Valores reales")
plt.ylabel("Valores predichos")
plt.title("Valores reales vs. predichos")
plt.show()

residuos = y_test - y_pred

plt.figure(figsize=(8,4))
sns.histplot(residuos, kde=True)
plt.title("Distribución de residuos")
plt.xlabel("Residuo")
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(x=residuos)
plt.title("Boxplot de residuos")
plt.xlabel("Residuo")
plt.show()

# 7. Guardado opcional
resultados = pd.DataFrame({
    "real": y_test,
    "predicho": y_pred,
    "residuo": residuos
})

resultados.to_csv("/workspace/housing_predicciones.csv", index=False)

print("Conclusión:")
print("Se ha cargado Housing.csv con ruta absoluta, se ha realizado EDA, preprocesamiento, codificación one-hot, estandarización, división train/test, entrenamiento de un modelo de regresión lineal, evaluación con MSE, MAE y R2, y visualización de predicciones y residuos.")
